# Ground Truth Generator

In [8]:
# ============================
# In-Domain Ground Truth Generator — Balanced 00/01/10/11
# Generates HDF5 with XOR circuit simulation data
# ============================

# ---- PATHS ----
PROJECT_DIR = "/content/drive/MyDrive/InDomainData/DataGeneration"
OUTDIR = "/content/drive/MyDrive/InDomainData/DataGeneration/Data"

# ---- CONFIGURATION ----
DATASET_TAG = "InDomainXOR"
TRIAL_LENGTH_MS = 100          # Duration per trial
TRIALS_PER_TYPE = 100          # Per pattern (00, 01, 10, 11) - total = 4×

# Stimulus timing guards
START_GUARD_MS = 5             # Earliest stim time
END_GUARD_MS = 39              # Latest = trial_length - 1 - end_guard

RNG_SEED = 123                 # For reproducibility (None = random)

# Optional weight overrides (None = use defaults)
WEIGHTS = None
# WEIGHTS = {
#     'A_to_C': 0.7, 'B_to_C': 0.7,
#     'A_to_D': 1.0, 'B_to_D': 1.0,
#     'D_to_E': 1.0, 'C_to_E': -1.0,
# }

# ---- Setup ----
import os, sys, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import h5py

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from NeuronDataWriter import NeuronDataWriter
from utils import build_in_domain_trials, run_single_in_domain_trial

# ---- Output path ----
gt_path = Path(OUTDIR) / f"{DATASET_TAG}_balanced_{4*TRIALS_PER_TYPE}trials_{TRIAL_LENGTH_MS}ms.hdf5"

print("Project dir:", PROJECT_DIR)
print("Output dir:", OUTDIR)
print("GT HDF5:", gt_path)

# ---- Validate stimulus window ----
low = int(START_GUARD_MS)
high = int(TRIAL_LENGTH_MS - 1 - END_GUARD_MS)
if not (0 <= low <= high < TRIAL_LENGTH_MS):
    raise ValueError(f"Invalid stim window [{low}, {high}] for trial_length={TRIAL_LENGTH_MS}")
print(f"Stim window: [{low}, {high}]")

# ---- Build balanced trial plan ----
trials = build_in_domain_trials(
    trials_per_type=TRIALS_PER_TYPE,
    trial_length_ms=TRIAL_LENGTH_MS,
    start_guard_ms=START_GUARD_MS,
    end_guard_ms=END_GUARD_MS,
    rng_seed=RNG_SEED,
)

plan_counts = Counter((t['a_bit'], t['b_bit']) for t in trials)
print("Pattern counts:", {f"{a}{b}": plan_counts[(a,b)] for a,b in [(0,0),(0,1),(1,0),(1,1)]})

# ---- Generate and write trials ----
if gt_path.exists():
    os.remove(gt_path)

writer = NeuronDataWriter(str(gt_path))

for idx, tr in enumerate(trials):
    nn, a_bit, b_bit, a_time, b_time = run_single_in_domain_trial(
        trial_def=tr,
        trial_length_ms=TRIAL_LENGTH_MS,
        weights=WEIGHTS,
    )
    writer.write_trial(
        neuron_network=nn,
        trial_idx=idx,
        trial_length_ms=TRIAL_LENGTH_MS,
        a_bit=a_bit,
        b_bit=b_bit,
        a_stim_time=a_time,
        b_stim_time=b_time,
    )
    if (idx + 1) % 200 == 0 or (idx + 1) == len(trials):
        print(f"Wrote trial {idx + 1}/{len(trials)}")

print("GT saved:", gt_path)

# ---- Verification ----
def verify_counts(h5path):
    """Check realized pattern distribution."""
    counts = Counter()
    with h5py.File(h5path, "r") as f:
        for gname in f.keys():
            meta = json.loads(f[gname].attrs['trial_meta'])
            counts[(meta['a_bit'], meta['b_bit'])] += 1
    return counts

def inspect_trials(h5path, n=2):
    """Display structure of first n trials."""
    with h5py.File(h5path, "r") as f:
        keys = sorted(f.keys(), key=lambda s: int(s.split('_')[-1]))
        for g in keys[:n]:
            d = f[g]["data"]
            cols = list(map(str, d.attrs["columns"]))
            print(f"[{g}] shape={d.shape}, cols={cols[:6]}... ({len(cols)} total)")

# Verify pattern distribution
realized = verify_counts(gt_path)
print("Realized counts:", {f"{a}{b}": realized[(a,b)] for a,b in [(0,0),(0,1),(1,0),(1,1)]})

# Sample inspection
inspect_trials(gt_path, n=2)

# Verify stimulus timing bounds
mins, maxs = defaultdict(lambda: float('inf')), defaultdict(lambda: float('-inf'))
with h5py.File(gt_path, "r") as f:
    for gname in f.keys():
        meta = json.loads(f[gname].attrs["trial_meta"])
        if meta["a_stim_time"] is not None:
            mins["A"] = min(mins["A"], meta["a_stim_time"])
            maxs["A"] = max(maxs["A"], meta["a_stim_time"])
        if meta["b_stim_time"] is not None:
            mins["B"] = min(mins["B"], meta["b_stim_time"])
            maxs["B"] = max(maxs["B"], meta["b_stim_time"])

for ch in ["A", "B"]:
    if mins[ch] != float('inf'):
        print(f"{ch}_stim range: [{mins[ch]}, {maxs[ch]}] within [{low}, {high}]")

print("\nGround truth generation complete!")
print(f"Output: {gt_path}")

Project dir: /content/drive/MyDrive/InDomainData/DataGeneration
Output dir: /content/drive/MyDrive/InDomainData/DataGeneration/Data
GT HDF5: /content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_balanced_400trials_100ms.hdf5
Stim window: [5, 60]
Pattern counts: {'00': 100, '01': 100, '10': 100, '11': 100}
Wrote trial 200/400
Wrote trial 400/400
GT saved: /content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_balanced_400trials_100ms.hdf5
Realized counts: {'00': 100, '01': 100, '10': 100, '11': 100}
[trial_0] shape=(100, 13), cols=['time', 'A_stim', 'B_stim', 'Neuron_A_spike_train', 'Neuron_A_membrane_potential', 'Neuron_B_spike_train']... (13 total)
[trial_1] shape=(100, 13), cols=['time', 'A_stim', 'B_stim', 'Neuron_A_spike_train', 'Neuron_A_membrane_potential', 'Neuron_B_spike_train']... (13 total)
A_stim range: [5, 60] within [5, 60]
B_stim range: [5, 60] within [5, 60]

Ground truth generation complete!
Output: /content/drive/MyDrive/InDomainData/DataGene

# Submission Generator

In [9]:
# ============================
# Submission Generator
#
# This script is designed to be modified for testing different circuit behaviors.
# Current configuration: C_to_E = +1.0 (excitatory) instead of -1.0 (inhibitory)
#
# TO EXPERIMENT WITH DIFFERENT BEHAVIORS:
#   1. Modify the WEIGHTS dictionary below
#   2. Update DATASET_TAG to reflect your experiment
#   3. Run to generate new submission dataset
#   4. Compare results with ground truth to test model learning
#
# ============================

# ---- USER PATHS ----
PROJECT_DIR = "/content/drive/MyDrive/InDomainData/DataGeneration"    # your .py modules location
OUTDIR      = "/content/drive/MyDrive/InDomainData/DataGeneration/Data"

# ---- DATASET CONFIG (Match GT for fair comparison) ----
DATASET_TAG      = "InDomainXOR_SUB"   # CHANGE THIS for each experiment
TRIAL_LENGTH_MS  = 100                 # Keep same as GT for consistency
TRIALS_PER_TYPE  = 100                 # Keep same as GT for consistency

# Stimulus timing guards (keep same as GT for controlled comparison)
START_GUARD_MS   = 5                   # earliest stim time
END_GUARD_MS     = 39                  # buffer from end (allows ~40ms response time)

RNG_SEED         = 123                 # Use same seed as GT for comparable trial structure

# ====== EXPERIMENTAL WEIGHTS - MODIFY HERE ======
# This is where you change circuit behavior for experiments
#
# DEFAULT XOR CIRCUIT (from GT):
#   A_to_C: 0.7 (subthreshold alone)
#   B_to_C: 0.7 (subthreshold alone)
#   A_to_D: 1.0 (suprathreshold)
#   B_to_D: 1.0 (suprathreshold)
#   D_to_E: 1.0 (excitatory)
#   C_to_E: -1.0 (inhibitory) ← Makes XOR work

WEIGHTS = {
    'A_to_C': 0.7,    # Tweak for different AND sensitivity
    'B_to_C': 0.7,    # Tweak for different AND sensitivity
    'A_to_D': 1.0,    # Tweak for different OR sensitivity
    'B_to_D': 1.0,    # Tweak for different OR sensitivity
    'D_to_E': -1.0,    # Try negative for inverted logic
    'C_to_E': 1.0,    # ← CHANGED FROM -1.0 (GT) TO +1.0 (excitatory)
}

GT_WEIGHTS = {
    'A_to_C': 0.7,
    'B_to_C': 0.7,
    'A_to_D': 1.0,
    'B_to_D': 1.0,
    'D_to_E': 1.0,
    'C_to_E': -1.0,  # This makes XOR work in GT
}

print("=" * 60)
print("SUBMISSION DATASET")
print(f"Experiment tag: {DATASET_TAG}")
print("Modified weights from GT:")
changes_found = False
for key in GT_WEIGHTS:
    gt_val = GT_WEIGHTS[key]
    exp_val = WEIGHTS[key]
    if exp_val != gt_val:
        print(f"  {key}: {gt_val} → {exp_val} (CHANGED)")
        changes_found = True
if not changes_found:
    print("  No changes - using GT weights")
print("=" * 60)

# ---- Imports & setup ----
import os, sys, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import h5py

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from NeuronDataWriter import NeuronDataWriter
from utils import build_in_domain_trials, run_single_in_domain_trial

# ---- Setup output file ----
sub_path = Path(OUTDIR) / f"{DATASET_TAG}_balanced_{4*TRIALS_PER_TYPE}trials_{TRIAL_LENGTH_MS}ms.hdf5"

print("Project dir    :", PROJECT_DIR)
print("Output dir     :", OUTDIR)
print("Submission HDF5:", sub_path)

# ---- Validate stimulus window (should match GT) ----
low  = int(START_GUARD_MS)
high = int(TRIAL_LENGTH_MS - 1 - END_GUARD_MS)
if not (0 <= low <= high < TRIAL_LENGTH_MS):
    raise ValueError(
        f"Invalid guards for T={TRIAL_LENGTH_MS}: low={low}, high={high} "
        f"(start_guard={START_GUARD_MS}, end_guard={END_GUARD_MS})"
    )
print(f"Stim window: [{low}, {high}] ms (should match GT for fair comparison)")

# ---- Build balanced trial schedule (use same process as GT) ----
# IMPORTANT: Using same seed as GT ensures comparable trial structure
# This isolates the effect of weight changes from trial variability
trials = build_in_domain_trials(
    trials_per_type=TRIALS_PER_TYPE,
    trial_length_ms=TRIAL_LENGTH_MS,
    start_guard_ms=START_GUARD_MS,
    end_guard_ms=END_GUARD_MS,
    rng_seed=RNG_SEED,  # Same seed → same trial order/timing as GT
)

plan_counts = Counter((t['a_bit'], t['b_bit']) for t in trials)
print("Planned patterns:", {f"{a}{b}": plan_counts[(a,b)] for a,b in [(0,0),(0,1),(1,0),(1,1)]})

# ---- Generate & write trials with EXPERIMENTAL weights ----
if sub_path.exists():
    print(f"Warning: Overwriting existing {sub_path.name}")
    os.remove(sub_path)

writer = NeuronDataWriter(str(sub_path))

print("\nGenerating with experimental circuit behavior...")
for idx, tr in enumerate(trials):
    # Run with EXPERIMENTAL weights (different from GT)
    nn, a_bit, b_bit, a_time, b_time = run_single_in_domain_trial(
        trial_def=tr,
        trial_length_ms=TRIAL_LENGTH_MS,
        weights=WEIGHTS,    # ← Using experimental weights
    )

    writer.write_trial(
        neuron_network=nn,
        trial_idx=idx,
        trial_length_ms=TRIAL_LENGTH_MS,
        a_bit=a_bit, b_bit=b_bit,
        a_stim_time=a_time, b_stim_time=b_time,
    )

    if (idx + 1) % 200 == 0 or (idx + 1) == len(trials):
        print(f"  wrote trial {idx + 1}/{len(trials)}")

print("\nSubmission saved:", sub_path)

# ---- Verification checks (keep detailed for debugging) ----
def realized_counts(h5path):
    """Count actual pattern distribution in generated file"""
    counts = Counter()
    with h5py.File(h5path, "r") as f:
        for gname in f.keys():
            meta = json.loads(f[gname].attrs['trial_meta'])
            counts[(meta['a_bit'], meta['b_bit'])] += 1
    return counts

def inspect_sample(h5path, n=2):
    """Show structure of first n trials for quick verification"""
    with h5py.File(h5path, "r") as f:
        keys = sorted(f.keys(), key=lambda s: int(s.split('_')[-1]))
        for g in keys[:n]:
            d = f[g]["data"]
            cols = list(map(str, d.attrs["columns"]))
            print(f"[{g}] shape={d.shape} cols={cols[:6]} ... (total {len(cols)})")

# Check that pattern distribution matches plan
rc = realized_counts(sub_path)
print("Realized patterns:", {f"{a}{b}": rc[(a,b)] for a,b in [(0,0),(0,1),(1,0),(1,1)]})

# Quick structure check
inspect_sample(sub_path, n=2)

# Verify stimulus times are within bounds (important for valid trials)
from math import inf
mins, maxs = defaultdict(lambda: inf), defaultdict(lambda: -inf)
with h5py.File(sub_path, "r") as f:
    for gname in f.keys():
        meta = json.loads(f[gname].attrs["trial_meta"])
        if meta["a_stim_time"] is not None:
            mins["A"] = min(mins["A"], meta["a_stim_time"])
            maxs["A"] = max(maxs["A"], meta["a_stim_time"])
        if meta["b_stim_time"] is not None:
            mins["B"] = min(mins["B"], meta["b_stim_time"])
            maxs["B"] = max(maxs["B"], meta["b_stim_time"])

for ch in ["A", "B"]:
    if mins[ch] != inf:
        print(f"{ch}_stim range: [{mins[ch]}, {maxs[ch]}] within [{low}, {high}]")

print("\nSubmission generation complete!")
print(f"Output file: {sub_path}")

SUBMISSION DATASET
Experiment tag: InDomainXOR_SUB
Modified weights from GT:
  D_to_E: 1.0 → -1.0 (CHANGED)
  C_to_E: -1.0 → 1.0 (CHANGED)
Project dir    : /content/drive/MyDrive/InDomainData/DataGeneration
Output dir     : /content/drive/MyDrive/InDomainData/DataGeneration/Data
Submission HDF5: /content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_SUB_balanced_400trials_100ms.hdf5
Stim window: [5, 60] ms (should match GT for fair comparison)
Planned patterns: {'00': 100, '01': 100, '10': 100, '11': 100}

Generating with experimental circuit behavior...
  wrote trial 200/400
  wrote trial 400/400

Submission saved: /content/drive/MyDrive/InDomainData/DataGeneration/Data/InDomainXOR_SUB_balanced_400trials_100ms.hdf5
Realized patterns: {'00': 100, '01': 100, '10': 100, '11': 100}
[trial_0] shape=(100, 13) cols=['time', 'A_stim', 'B_stim', 'Neuron_A_spike_train', 'Neuron_A_membrane_potential', 'Neuron_B_spike_train'] ... (total 13)
[trial_1] shape=(100, 13) cols=['time', 'A_